# Лабораторна робота №5 (MLP / Backprop) — Wine Quality (red)




## 0. Setup

In [ ]:
# --- imports ---
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt

# reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Завантаження і препроцесинг даних

In [ ]:
def load_and_preprocess_data(test_size=0.15, val_size=0.15):
    '''
    Кроки:
    1) Завантажити CSV Wine Quality (red)
    2) Відокремити X та y
    3) Нормалізувати StandardScaler
    4) Поділ train/val/test
    5) Конвертувати в torch.Tensor
    '''
    url_red = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    df = pd.read_csv(url_red, sep=";")  # UCI uses ';'

    X = df.drop(columns=["quality"]).values
    y_raw = df["quality"].values

    # Label encoding -> 0..K-1
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    class_names = [str(c) for c in le.classes_]

    # Split train vs temp (val+test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(test_size + val_size), random_state=SEED, stratify=y
    )

    # Split val vs test from temp
    val_ratio_in_temp = val_size / (test_size + val_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1 - val_ratio_in_temp),
        random_state=SEED, stratify=y_temp
    )

    # Normalize from train stats only
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # to torch tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_val_t   = torch.tensor(X_val, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test, dtype=torch.float32)

    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_val_t   = torch.tensor(y_val, dtype=torch.long)
    y_test_t  = torch.tensor(y_test, dtype=torch.long)

    return X_train_t, y_train_t, X_val_t, y_val_t, X_test_t, y_test_t, class_names

X_train, y_train, X_val, y_val, X_test, y_test, class_names = load_and_preprocess_data()

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)
print("Classes:", class_names)

## 2. Функції активації (forward + derivative)

In [ ]:
class ActivationFunctions:
    @staticmethod
    def relu(x, derivative=False):
        if derivative:
            return (x > 0).float()
        return torch.clamp(x, min=0.0)

    @staticmethod
    def tanh(x, derivative=False):
        if derivative:
            return 1 - torch.tanh(x)**2
        return torch.tanh(x)

    @staticmethod
    def softmax(x):
        # stable softmax over rows (classes)
        x_shift = x - x.max(dim=0, keepdim=True).values
        exp_x = torch.exp(x_shift)
        return exp_x / exp_x.sum(dim=0, keepdim=True)

## 3. Функція втрат Cross-Entropy + похідна softmax+CE

In [ ]:
class LossFunctions:
    @staticmethod
    def categorical_cross_entropy(y_pred, y_true, derivative=False):
        '''
        y_pred: (batch, num_classes) probabilities
        y_true: (batch,) integer labels
        '''
        eps = 1e-9
        batch_size = y_pred.shape[0]

        if derivative:
            # grad for logits in softmax+CE combo:
            # dZ = (y_pred - y_true_onehot) / batch
            grad = y_pred.clone()
            grad[range(batch_size), y_true] -= 1.0
            return grad / batch_size

        # forward loss
        correct_probs = y_pred[range(batch_size), y_true]
        loss = -torch.log(correct_probs + eps).mean()
        return loss

## 4. Custom MLP з нуля (без autograd)

In [ ]:
class CustomMLP:
    def __init__(self, layer_sizes, activations, learning_rate=0.01):
        '''
        layer_sizes: [input, h1, h2, ..., output]
        activations: list like ["relu","tanh", ...] for hidden layers
        '''
        self.layer_sizes = layer_sizes
        self.activations = activations
        self.lr = learning_rate
        self.parameters = {}
        self._init_params()

    def _init_params(self):
        for i in range(1, len(self.layer_sizes)):
            in_dim  = self.layer_sizes[i-1]
            out_dim = self.layer_sizes[i]
            # Xavier init
            limit = np.sqrt(6.0 / (in_dim + out_dim))
            W = torch.empty(out_dim, in_dim).uniform_(-limit, limit)
            b = torch.zeros(out_dim, 1)
            self.parameters[f"W{i}"] = W
            self.parameters[f"b{i}"] = b

    def _act(self, name, x, derivative=False):
        if name == "relu":
            return ActivationFunctions.relu(x, derivative=derivative)
        if name == "tanh":
            return ActivationFunctions.tanh(x, derivative=derivative)
        raise ValueError(f"Unknown activation {name}")

    def forward(self, X):
        '''
        X: (batch, features)
        cache stores A and Z in column-major form
        '''
        cache = {}
        A_prev = X.T  # (in_dim, batch)
        cache["A0"] = A_prev

        L = len(self.layer_sizes) - 1
        for i in range(1, L + 1):
            W = self.parameters[f"W{i}"]
            b = self.parameters[f"b{i}"]
            Z = W @ A_prev + b  # (out_dim, batch)
            cache[f"Z{i}"] = Z

            if i < L:
                act_name = self.activations[i-1]
                A = self._act(act_name, Z)
            else:
                A = ActivationFunctions.softmax(Z)

            cache[f"A{i}"] = A
            A_prev = A

        return cache[f"A{L}"].T, cache  # (batch, classes)

    def backward(self, y_pred, y_true, cache):
        '''
        y_pred: (batch, classes)
        y_true: (batch,)
        '''
        L = len(self.layer_sizes) - 1

        # dZ for output
        dZ = LossFunctions.categorical_cross_entropy(y_pred, y_true, derivative=True).T  # (classes, batch)

        for i in range(L, 0, -1):
            A_prev = cache[f"A{i-1}"]  # (prev_dim, batch)
            W = self.parameters[f"W{i}"]

            dW = dZ @ A_prev.T
            db = dZ.sum(dim=1, keepdim=True)

            # update
            self.parameters[f"W{i}"] -= self.lr * dW
            self.parameters[f"b{i}"] -= self.lr * db

            if i > 1:
                dA_prev = W.T @ dZ
                Z_prev = cache[f"Z{i-1}"]
                act_name = self.activations[i-2]
                dZ = dA_prev * self._act(act_name, Z_prev, derivative=True)

    def predict(self, X):
        y_pred, _ = self.forward(X)
        return torch.argmax(y_pred, dim=1)

    def fit(self, X_train, y_train, X_val, y_val, epochs=300, batch_size=64):
        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

        n = X_train.shape[0]
        for ep in range(1, epochs + 1):
            idx = torch.randperm(n)
            X_sh = X_train[idx]
            y_sh = y_train[idx]

            for start in range(0, n, batch_size):
                end = start + batch_size
                xb = X_sh[start:end]
                yb = y_sh[start:end]

                y_pred, cache = self.forward(xb)
                self.backward(y_pred, yb, cache)

            with torch.no_grad():
                train_pred_probs, _ = self.forward(X_train)
                val_pred_probs, _   = self.forward(X_val)

                tr_loss = LossFunctions.categorical_cross_entropy(train_pred_probs, y_train)
                va_loss = LossFunctions.categorical_cross_entropy(val_pred_probs, y_val)

                tr_acc = (train_pred_probs.argmax(1) == y_train).float().mean()
                va_acc = (val_pred_probs.argmax(1) == y_val).float().mean()

            history["train_loss"].append(tr_loss.item())
            history["val_loss"].append(va_loss.item())
            history["train_acc"].append(tr_acc.item())
            history["val_acc"].append(va_acc.item())

            if ep % 50 == 0 or ep == 1:
                print(f"Epoch {ep:4d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")

        return history

## 5. Навчання Custom MLP

In [ ]:
input_size = X_train.shape[1]
num_classes = len(class_names)

custom_model = CustomMLP(
    layer_sizes=[input_size, 64, 32, num_classes],
    activations=["relu", "tanh"],
    learning_rate=0.01
)

history_custom = custom_model.fit(X_train, y_train, X_val, y_val, epochs=400, batch_size=64)

## 6. PyTorch MLP для порівняння

In [ ]:
class PyTorchMLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, output_size))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def train_pytorch_model(model, X_train, y_train, X_val, y_val, epochs=400, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for ep in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train)
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            tr_logits = model(X_train)
            va_logits = model(X_val)

            tr_loss = criterion(tr_logits, y_train)
            va_loss = criterion(va_logits, y_val)

            tr_acc = (tr_logits.argmax(1) == y_train).float().mean()
            va_acc = (va_logits.argmax(1) == y_val).float().mean()

        history["train_loss"].append(tr_loss.item())
        history["val_loss"].append(va_loss.item())
        history["train_acc"].append(tr_acc.item())
        history["val_acc"].append(va_acc.item())

        if ep % 50 == 0 or ep == 1:
            print(f"[PT] Epoch {ep:4d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")

    return history

pt_model = PyTorchMLP(input_size, [64, 32], num_classes)
history_pt = train_pytorch_model(pt_model, X_train, y_train, X_val, y_val)

## 7. Графіки навчання (loss/accuracy)

In [ ]:
def plot_training_history(hc, hp):
    epochs = range(1, len(hc["train_loss"]) + 1)

    plt.figure()
    plt.plot(epochs, hc["train_loss"], label="Custom train")
    plt.plot(epochs, hc["val_loss"], label="Custom val")
    plt.plot(epochs, hp["train_loss"], label="PyTorch train")
    plt.plot(epochs, hp["val_loss"], label="PyTorch val")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(epochs, hc["train_acc"], label="Custom train")
    plt.plot(epochs, hc["val_acc"], label="Custom val")
    plt.plot(epochs, hp["train_acc"], label="PyTorch train")
    plt.plot(epochs, hp["val_acc"], label="PyTorch val")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

plot_training_history(history_custom, history_pt)

## 8. Оцінка на тесті + Confusion Matrix

In [ ]:
y_pred_custom = custom_model.predict(X_test).numpy()

pt_model.eval()
with torch.no_grad():
    y_pred_pt = pt_model(X_test).argmax(1).numpy()

print("Custom test accuracy:", accuracy_score(y_test.numpy(), y_pred_custom))
print("PyTorch test accuracy:", accuracy_score(y_test.numpy(), y_pred_pt))

def plot_confusion_matrix(y_true, y_pred_custom, y_pred_pt, class_names):
    cm_c = confusion_matrix(y_true, y_pred_custom)
    cm_p = confusion_matrix(y_true, y_pred_pt)

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay(cm_c, display_labels=class_names).plot(ax=ax[0], values_format="d")
    ax[0].set_title("Custom MLP")

    ConfusionMatrixDisplay(cm_p, display_labels=class_names).plot(ax=ax[1], values_format="d")
    ax[1].set_title("PyTorch MLP")

    plt.show()

plot_confusion_matrix(y_test.numpy(), y_pred_custom, y_pred_pt, class_names)